In [ ]:
 # Step 01-02
import requests

url = "https://www.rba.gov.au/statistics/tables/csv/f11.1-data.csv"

response = requests.get(url, timeout=30)
response.raise_for_status()

print("HTTP status:", response.status_code)
print("Downloaded bytes:", len(response.content))
print("First 300 characters:")
print(response.content[:300].decode("utf-8-sig", errors="replace"))


In [ ]:


# Step 03.1 — Duplicate Source Detection
# Evaluate the current HTTP source before downstream processing

import hashlib
from pyspark.sql import functions as F

source_url = url
fingerprint = hashlib.sha256(response.content).hexdigest()

registry_table = "default.rba_fx_v2_pipeline_runs"

if spark.catalog.tableExists(registry_table):
    successful_runs = (
        spark.table(registry_table)
        .filter(
            (F.col("source") == source_url)
            & (F.col("source_fingerprint") == fingerprint)
            & (F.col("status") == "SUCCESS")
        )
        .count()
    )
else:
    successful_runs = 0

should_process = successful_runs == 0

print("Step 03 — Duplicate Source Detection")
print("Matching successful runs:", successful_runs)
print("Should process:", should_process)


In [ ]:


# Step 03.2 — Register or Resume Run
# Register RUNNING before processing downstream steps

import uuid
from pyspark.sql import functions as F

registry_table = "default.rba_fx_v2_pipeline_runs"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {registry_table} (
    run_id STRING,
    source STRING,
    source_fingerprint STRING,
    status STRING,
    staged_count BIGINT,
    valid_count BIGINT,
    invalid_count BIGINT,
    new_count BIGINT,
    changed_count BIGINT,
    unchanged_count BIGINT,
    recorded_at TIMESTAMP
) USING DELTA
""")

# Reuse an unfinished run for the same fingerprint
existing = (
    spark.table(registry_table)
    .filter(
        (F.col("source") == source_url)
        & (F.col("source_fingerprint") == fingerprint)
    )
    .orderBy(F.col("recorded_at").desc())
    .limit(1)
    .collect()
)

if existing and existing[0]["status"] == "SUCCESS":
    should_process = False
    run_id = existing[0]["run_id"]
elif existing:
    run_id = existing[0]["run_id"]
else:
    run_id = str(uuid.uuid4())

    new_run = spark.createDataFrame(
        [(run_id, source_url, fingerprint, "RUNNING")],
        "run_id STRING, source STRING, "
        "source_fingerprint STRING, status STRING"
    ).withColumn(
        "recorded_at", F.current_timestamp()
    )

    new_run.createOrReplaceTempView("new_pipeline_run")

    spark.sql(f"""
    INSERT INTO {registry_table} (
        run_id, source, source_fingerprint,
        status, recorded_at
    )
    SELECT run_id, source, source_fingerprint,
           status, recorded_at
    FROM new_pipeline_run
    """)

print("Step 03 — Run Registry")
print("Run ID:", run_id)

# Stop the notebook when this source was already processed
if not should_process:
    print("SKIP: Source already processed successfully")
    dbutils.notebook.exit(
        "SKIPPED: Duplicate source fingerprint"
    )

print("PROCESS: Starting pipeline run", run_id)



In [ ]:
# Step 03.3
import hashlib
from datetime import datetime, timezone
from pyspark.sql import functions as F

# Reuse the HTTP response from the previous cell
raw_text = response.content.decode("utf-8-sig")
source_hash = hashlib.sha256(response.content).hexdigest()

# Preserve the original source as individual lines
lines = raw_text.splitlines()

bronze_df = spark.createDataFrame(
    [(i + 1, line) for i, line in enumerate(lines)],
    ["line_number", "raw_line"]
).withColumn(
    "source_url", F.lit(url)
).withColumn(
    "source_sha256", F.lit(source_hash)
).withColumn(
    "ingested_at", F.current_timestamp()
).withColumn(
    "run_id", F.lit(run_id)
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(
    "default.rba_fx_v2_bronze"
)

print("Source SHA256:", source_hash)
print("Raw lines:", bronze_df.count())

display(
    spark.table("default.rba_fx_v2_bronze")
         .orderBy("line_number")
         .limit(10)
)


In [ ]:

# Step 04 — Remove RBA Metadata
# Replicates the existing n8n node logic

source_lines = raw_text.splitlines()

header_index = next(
    (
        i for i, line in enumerate(source_lines)
        if line.lstrip("\ufeff").startswith("Series ID,")
    ),
    None
)

if header_index is None:
    raise ValueError("Could not find the RBA Series ID header")

cleaned_csv = "\n".join(source_lines[header_index:]).strip()

print("Step 04: SUCCESS")
print("Header found at row:", header_index)
print("Metadata lines removed:", header_index)
print("First line:", cleaned_csv.splitlines()[0])
print("Cleaned lines:", len(cleaned_csv.splitlines()))


In [ ]:

# Step 05 — Parse CSV Rows
# Replicates the n8n "Parse CSV Rows" node

import csv
import io

matrix = [
    row for row in csv.reader(io.StringIO(cleaned_csv))
    if any(str(value).strip() for value in row)
]

if len(matrix) < 2:
    raise ValueError("CSV contains a header but no data rows")

headers = [h.lstrip("\ufeff").strip() for h in matrix[0]]

if headers[0] != "Series ID":
    raise ValueError(f"Unexpected first header: {headers[0]}")

if any(not h for h in headers):
    raise ValueError("Blank CSV header detected")

if len(headers) != len(set(headers)):
    raise ValueError("Duplicate CSV headers detected")

parsed_records = []

for data_index, values in enumerate(matrix[1:], start=1):
    if len(values) != len(headers):
        raise ValueError(
            f"Malformed CSV at data row {data_index}: "
            f"expected {len(headers)} fields, got {len(values)}"
        )

    record = dict(zip(headers, values))
    record["source_row_no"] = data_index
    record["source"] = url
    parsed_records.append(record)

print("Step 05: SUCCESS")
print("Columns:", len(headers))
print("Parsed data rows:", len(parsed_records))
print("First data row:", parsed_records[0])
print("Last data row:", parsed_records[-1])


In [ ]:

# Step 06 — Normalize FX Rates to Long Format
# Replicates the original n8n normalization node using PySpark

from pyspark.sql import functions as F
from pyspark.sql import types as T

series_columns = [
    h for h in headers if h != "Series ID"
]

# Build a Spark DataFrame from Step 05 output
wide_rows = [
    (
        r["source_row_no"],
        r["Series ID"],
        *[r[h] for h in series_columns]
    )
    for r in parsed_records
]

wide_schema = T.StructType([
    T.StructField("source_row_no", T.IntegerType(), False),
    T.StructField("raw_rate_date", T.StringType(), True),
    *[
        T.StructField(h, T.StringType(), True)
        for h in series_columns
    ]
])

wide_df = spark.createDataFrame(wide_rows, wide_schema)

# Convert 23 series columns into long-form records
rates_array = F.array(*[
    F.struct(
        F.lit(h).alias("series_id"),
        F.col(f"`{h}`").alias("raw_rate")
    )
    for h in series_columns
])

normalized_df = (
    wide_df
    .select(
        "source_row_no",
        "raw_rate_date",
        F.explode(rates_array).alias("fx")
    )
    .select(
        "source_row_no",
        "raw_rate_date",
        F.col("fx.series_id").alias("series_id"),
        F.col("fx.raw_rate").alias("raw_rate")
    )
    .withColumn("source", F.lit(url))
)

actual_count = normalized_df.count()
expected_count = len(parsed_records) * len(series_columns)

assert actual_count == expected_count, (
    f"Expected {expected_count}, got {actual_count}"
)

print("Step 06: SUCCESS")
print("Source rows:", len(parsed_records))
print("FX series:", len(series_columns))
print("Normalized records:", actual_count)

display(normalized_df.limit(10))


In [ ]:

# Step 07 — Package Staging Batch / Delta Staging

from pyspark.sql import functions as F

staging_df = normalized_df.select(
    "source_row_no",
    "raw_rate_date",
    "series_id",
    "raw_rate",
    "source"
).withColumn(
    "run_id", F.lit(run_id)
)

record_count = staging_df.count()

if record_count == 0:
    raise ValueError("No normalized records received")

if record_count != expected_count:
    raise ValueError(
        f"Expected {expected_count}, got {record_count}"
    )

# Persist normalized records in a Delta staging table
staging_df.write.format("delta").mode("overwrite").saveAsTable(
    "default.rba_fx_v2_staging"
)

# Verify persisted records
saved_df = spark.table("default.rba_fx_v2_staging")
saved_count = saved_df.count()

assert saved_count == record_count, (
    f"Staging count mismatch: {saved_count} != {record_count}"
)

print("Step 07: SUCCESS")
print("Normalized records:", record_count)
print("Persisted staging records:", saved_count)
print("Delta table: default.rba_fx_v2_staging")

display(saved_df.limit(10))


In [ ]:

# Step 08 — SQL Data Validation
# Based on the original n8n PostgreSQL validation rules

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Validate only records belonging to the current pipeline run
staging = (
    spark.table("default.rba_fx_v2_staging")
    .filter(F.col("run_id") == run_id)
)

checked = (
    staging
    .withColumn("clean_date", F.trim("raw_rate_date"))
    .withColumn("clean_series", F.trim("series_id"))
    .withColumn("clean_rate", F.trim("raw_rate"))
    .withColumn(
        "date_format_ok",
        F.coalesce(
            F.col("clean_date").rlike(
                r"^[0-9]{2}-[A-Za-z]{3}-[0-9]{4}$"
            ),
            F.lit(False)
        )
    )
    .withColumn(
        "parsed_date",
        F.to_date(
            F.expr(
                "try_to_timestamp(clean_date, 'dd-MMM-yyyy')"
            )
        )
    )
    .withColumn(
        "parsed_rate",
        F.expr("try_cast(clean_rate AS DECIMAL(20,8))")
    )
)

non_finite = [
    "NAN", "INFINITY", "+INFINITY", "-INFINITY",
    "INF", "+INF", "-INF"
]

key_window = Window.partitionBy(
    "source", "parsed_date", "clean_series"
)

checked = checked.withColumn(
    "business_key_count",
    F.when(
        F.col("parsed_date").isNotNull()
        & F.col("clean_series").isNotNull()
        & (F.col("clean_series") != ""),
        F.count("*").over(key_window)
    ).otherwise(F.lit(0))
)

# Each condition corresponds to an original n8n error code
rules = {
    "MISSING_DATE":
        F.col("clean_date").isNull() | (F.col("clean_date") == ""),
    "INVALID_DATE_FORMAT":
        F.col("clean_date").isNotNull()
        & (F.col("clean_date") != "")
        & ~F.col("date_format_ok"),
    "INVALID_DATE_VALUE":
        F.col("date_format_ok")
        & F.col("parsed_date").isNull(),
    "MISSING_SERIES_ID":
        F.col("clean_series").isNull() | (F.col("clean_series") == ""),
    "INVALID_SERIES_ID":
        F.col("clean_series").isNotNull()
        & (F.col("clean_series") != "")
        & ~F.col("clean_series").rlike(r"^FXR[A-Z0-9]+$"),
    "MISSING_RATE":
        F.col("clean_rate").isNull() | (F.col("clean_rate") == ""),
    "NON_FINITE_RATE":
        F.upper(F.col("clean_rate")).isin(non_finite),
    "INVALID_RATE":
        F.col("clean_rate").isNotNull()
        & (F.col("clean_rate") != "")
        & ~F.upper(F.col("clean_rate")).isin(non_finite)
        & F.col("parsed_rate").isNull(),
    "NON_POSITIVE_RATE":
        F.col("parsed_rate") <= 0,
    "DUPLICATE_BUSINESS_KEY":
        F.col("business_key_count") > 1
}

for code, condition in rules.items():
    checked = checked.withColumn(
        code, F.coalesce(condition, F.lit(False))
    )

error_columns = list(rules.keys())

checked = checked.withColumn(
    "is_valid",
    ~F.expr(
        " OR ".join(f"`{code}`" for code in error_columns)
    )
)


total_count = checked.count()
valid_count = checked.filter("is_valid").count()
invalid_count = total_count - valid_count

print("Step 08: SUCCESS")
print("Staged:", total_count)
print("Valid:", valid_count)
print("Invalid:", invalid_count)
print("Reconciliation:", total_count == valid_count + invalid_count)

display(
    checked.agg(
        *[
            F.sum(F.col(code).cast("int")).alias(code)
            for code in error_columns
        ]
    )
)


In [ ]:

# Step 09 — Exception Logging
# Rebuilds the original n8n dq_exceptions output

from pyspark.sql import functions as F

error_details = {
    "MISSING_DATE": "raw_rate_date is null or blank",
    "INVALID_DATE_FORMAT": "Expected date format DD-Mon-YYYY",
    "INVALID_DATE_VALUE": "Date text matches the expected format but is not a valid date",
    "MISSING_SERIES_ID": "series_id is null or blank",
    "INVALID_SERIES_ID": "Unexpected RBA series_id format",
    "MISSING_RATE": "raw_rate is null or blank",
    "NON_FINITE_RATE": "Rate is non-finite",
    "INVALID_RATE": "Rate cannot be converted to NUMERIC(20,8)",
    "NON_POSITIVE_RATE": "Rate must be greater than zero",
    "DUPLICATE_BUSINESS_KEY": "Duplicate (source, rate_date, series_id) exists inside this source run"
}

# Each failed rule creates one exception record.
# A single staging record may have multiple error codes.
exception_array = F.array(*[
    F.when(
        F.col(code),
        F.struct(
            F.lit(code).alias("error_code"),
            F.lit(detail).alias("error_detail")
        )
    )
    for code, detail in error_details.items()
])

exceptions_df = (
    checked
    .withColumn("run_id", F.lit(run_id))
    .select(
        "run_id",
        "source",
        "source_row_no",
        "raw_rate_date",
        "series_id",
        "raw_rate",
        F.explode(
            F.filter(exception_array, lambda x: x.isNotNull())
        ).alias("exception")
    )
    .select(
        "run_id",
        "source",
        "source_row_no",
        "raw_rate_date",
        "series_id",
        "raw_rate",
        "exception.error_code",
        "exception.error_detail"
    )
)

# Persist exceptions as a Delta table
exceptions_df.write.format("delta").mode("overwrite").saveAsTable(
    "default.rba_fx_v2_dq_exceptions"
)

saved_exceptions = (
    spark.table("default.rba_fx_v2_dq_exceptions")
    .filter(F.col("run_id") == run_id)
)

detected_errors = saved_exceptions.count()

invalid_records = (
    saved_exceptions
    .select("source", "source_row_no", "series_id")
    .distinct()
    .count()
)

assert invalid_records == invalid_count, (
    f"Invalid record mismatch: {invalid_records} != {invalid_count}"
)

print("Step 09: SUCCESS")
print("Invalid records:", invalid_records)
print("Detected errors:", detected_errors)
print("Delta table: default.rba_fx_v2_dq_exceptions")

display(
    saved_exceptions
    .groupBy("error_code")
    .count()
    .orderBy(F.desc("count"))
)


In [ ]:

# Step 10 — Build Validated Dataset and Classify
# Matches n8n: NEW / CHANGED / UNCHANGED

from pyspark.sql import functions as F

# Create an empty master table only if it does not exist.
# Do not overwrite an existing master.
spark.sql("""
CREATE TABLE IF NOT EXISTS default.rba_fx_v2_master (
    source STRING,
    rate_date DATE,
    series_id STRING,
    rate DECIMAL(20,8)
) USING DELTA
""")

# Accept only records that passed Step 08 validation
valid_df = (
    checked
    .filter(F.col("is_valid"))
    .select(
        "source",
        "source_row_no",
        F.col("parsed_date").alias("rate_date"),
        F.col("clean_series").alias("series_id"),
        F.col("parsed_rate").alias("rate")
    )
    .withColumn("run_id", F.lit(run_id))
)

master_df = spark.table("default.rba_fx_v2_master")

# Original n8n business key:
# source + rate_date + series_id
join_condition = (
    (F.col("v.source") == F.col("m.source")) &
    (F.col("v.rate_date") == F.col("m.rate_date")) &
    (F.col("v.series_id") == F.col("m.series_id"))
)

classified_df = (
    valid_df.alias("v")
    .join(master_df.alias("m"), join_condition, "left")
    .select(
        "v.*",
        F.when(
            F.col("m.source").isNull(), "NEW"
        ).when(
            ~F.col("v.rate").eqNullSafe(F.col("m.rate")),
            "CHANGED"
        ).otherwise("UNCHANGED").alias("load_action")
    )
)

# Persist classification results
classified_df.write.format("delta").mode("overwrite").saveAsTable(
    "default.rba_fx_v2_validated"
)

print("Step 10: SUCCESS")

display(
    spark.table("default.rba_fx_v2_validated")
    .groupBy("load_action")
    .count()
    .orderBy("load_action")
)


In [ ]:

# Step 11 — Conditional Upsert
# Replicates n8n conditional upsert using Delta MERGE

from pyspark.sql import functions as F

candidates = (
    spark.table("default.rba_fx_v2_validated")
    .filter(        
            (F.col("run_id") == run_id)
            & F.col("load_action").isin("NEW", "CHANGED"))
    .select("source", "rate_date", "series_id", "rate")
)

candidate_count = candidates.count()

candidates.createOrReplaceTempView("rba_merge_candidates")

spark.sql("""
MERGE INTO default.rba_fx_v2_master AS target
USING rba_merge_candidates AS source
ON  target.source = source.source
AND target.rate_date = source.rate_date
AND target.series_id = source.series_id

WHEN MATCHED AND NOT (target.rate <=> source.rate)
THEN UPDATE SET
    target.rate = source.rate

WHEN NOT MATCHED
THEN INSERT (source, rate_date, series_id, rate)
VALUES (
    source.source,
    source.rate_date,
    source.series_id,
    source.rate
)
""")

master_count = spark.table("default.rba_fx_v2_master").count()

print("Step 11: SUCCESS")
print("Upsert candidates:", candidate_count)
print("Master records:", master_count)
# Verify initial load without a hard-coded record count
assert master_count == candidate_count, (
    f"Master count mismatch: {master_count} != {candidate_count}"
)

print("Step 11 verification: PASS")


In [ ]:

# Step 12 — Run Summary & Reconciliation
# Verify initial-load counts against persisted Delta tables

from pyspark.sql import functions as F

staged = spark.table("default.rba_fx_v2_staging").count()

exceptions = spark.table("default.rba_fx_v2_dq_exceptions")

invalid = (
    exceptions
    .select("source", "source_row_no", "series_id")
    .distinct()
    .count()
)

validated = spark.table("default.rba_fx_v2_validated")

valid = validated.count()

action_counts = {
    row["load_action"]: row["count"]
    for row in validated.groupBy("load_action").count().collect()
}

new_count = action_counts.get("NEW", 0)
changed_count = action_counts.get("CHANGED", 0)
unchanged_count = action_counts.get("UNCHANGED", 0)

master_count = spark.table("default.rba_fx_v2_master").count()

checks = {
    "staging_reconciled": staged == valid + invalid,
    "classification_reconciled":
        valid == new_count + changed_count + unchanged_count,
    "master_count_covers_current_batch":
        master_count >= valid,
    "load_actions_valid":
        new_count >= 0
        and changed_count >= 0
        and unchanged_count >= 0
}

print("Step 12 — Run Summary & Reconciliation")
print("Staged:", staged)
print("Valid:", valid)
print("Invalid:", invalid)
print("NEW:", new_count)
print("CHANGED:", changed_count)
print("UNCHANGED:", unchanged_count)
print("Master records:", master_count)

print("\nReconciliation checks:")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

assert all(checks.values()), "Reconciliation failed"
master_check = (
    validated.alias("v")
    .join(
        spark.table("default.rba_fx_v2_master").alias("m"),
        ["source", "rate_date", "series_id"],
        "left"
    )
    .filter(
        F.col("m.source").isNull()
        | ~F.col("v.rate").eqNullSafe(F.col("m.rate"))
    )
    .count()
)

assert master_check == 0
print("Master key/value reconciliation: PASS")
print("\nStep 12: PASS")


# Step 12 — Mark pipeline run as SUCCESS
# Execute only after all reconciliation checks pass

assert all(checks.values()), "Cannot complete a failed run"

from pyspark.sql import functions as F

registry_table = "default.rba_fx_v2_pipeline_runs"

spark.sql(f"""
UPDATE {registry_table}
SET
    status = 'SUCCESS',
    staged_count = {staged},
    valid_count = {valid},
    invalid_count = {invalid},
    new_count = {new_count},
    changed_count = {changed_count},
    unchanged_count = {unchanged_count}
WHERE run_id = '{run_id}'
  AND status = 'RUNNING'
""")

completed = (
    spark.table(registry_table)
    .filter(
        (F.col("run_id") == run_id)
        & (F.col("status") == "SUCCESS")
    )
    .count()
)

assert completed == 1, "Run completion was not recorded"

print("Run Registry status: SUCCESS")
print("Step 12 — Completion: PASS")


In [ ]:

# Step 13 — Historical Monitoring
# Read-only monitoring of persisted pipeline runs

monitoring_df = spark.sql("""
SELECT
    run_id,
    source,
    SUBSTRING(source_fingerprint, 1, 12) AS fingerprint_prefix,
    status,
    staged_count,
    valid_count,
    invalid_count,
    ROUND(
        100.0 * invalid_count / NULLIF(staged_count, 0),
        2
    ) AS invalid_rate_pct,
    new_count,
    changed_count,
    unchanged_count,
    CASE
        WHEN staged_count = valid_count + invalid_count
        THEN 'PASS'
        ELSE 'FAIL'
    END AS count_reconciliation,
    recorded_at
FROM default.rba_fx_v2_pipeline_runs
ORDER BY recorded_at DESC
""")

print("Step 13 — Historical Monitoring")
print("Recorded runs:", monitoring_df.count())

display(monitoring_df)

assert monitoring_df.filter(
    "count_reconciliation = 'FAIL'"
).count() == 0

print("Step 13: PASS")
